**Content Produced by UF Signal Processing Society**

**Authors: Raul Valle & Contributors**

# Classical Forecasting: AR, MA & ARIMA

The linear baseline that beat the LSTM in the [RNN workshop](./Intro_RNN.ipynb) finally gets its own theory: AR and MA models, the ACF/PACF fingerprints that identify them, differencing for trends, and honest forecast intervals — all implemented from scratch.

## 1. Pre-requisites

- [Statistical Signal Processing](../Intro_DSP/Statistical_Signal_Processing.ipynb) S1 (WSS, autocorrelation).
- [Linear Algebra](../Intro_Math/Linear_Algebra/Linear_Algebra.ipynb) S2 (least squares).

In [1]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import signal as sig
rng = np.random.default_rng(0)

---
### 🕐 Session 1 of 2 — *AR & MA Models and Their Fingerprints* (~40 min)
**Goal:** read ACF/PACF plots to identify a model, then fit it by Yule-Walker/least squares.
**Builds on:** [Statistical SP](../Intro_DSP/Statistical_Signal_Processing.ipynb) S1. &nbsp; **Feeds into:** Session 2 (ARIMA & forecasting).

---

<details>
<summary>🎓 <b>Teacher notes — Session 1: AR & MA Models and Their Fingerprints</b></summary>

**Timing (~40 min).** 10 min the two kinds of memory · 8 min ACF/PACF and why the fingerprints are mirror images · 10 min the fingerprint demo · 8 min fitting · 4 min buffer.

**Board first — lean on what they already know.** This room has done [Filter Design](../Intro_DSP/Filter_Design.ipynb), so do not introduce AR and MA as statistics. AR is an all-pole IIR filter driven by white noise; MA is an FIR filter driven by white noise. Every property follows from that: MA's memory is a conveyor belt that runs out after $q$ taps, so its ACF must die at lag $q$; AR's memory recirculates through feedback, so its ACF decays forever without ever quite reaching zero. Students who see it as filtering derive the fingerprint table instead of memorising it.

**The PACF needs a plain-language gloss.** "Correlation at lag $k$ after regressing out lags $1..k{-}1$" is precise and nearly useless on first hearing. Say it as: your grandfather's height correlates with yours, but almost entirely *through* your father — the PACF asks what is left once you have already accounted for the intermediate generations. For an AR($p$) nothing is left past lag $p$, because the model literally says only $p$ lags feed in directly. That is why the PACF cuts where it does.

**Misconception.** "The ACF cutting off proves it's an MA process." It is evidence, not proof, and on real data the plots are far muddier than the ones this cell produces — 3000 clean samples from a known generator is the easy case. Say plainly that Box–Jenkins identification is a craft with a large judgement component, and that modern practice usually fits several orders and compares them on held-out data or by AIC rather than squinting at stems. Students who go on to real series will otherwise think they are doing it wrong.

**Ask the room.** Before running the fit cell: "we have two different estimators for the same AR(2). Should they give the same answer?" They should agree closely but not exactly — Yule–Walker solves the population normal equations using estimated autocorrelations, least squares minimises in-sample prediction error. Asymptotically equivalent, finite-sample different. This distinction pays off in Session 2, where the *forecast* is what we actually care about.

**Point at the confidence band.** The grey band is $\pm 1.96/\sqrt{N}$, the null-hypothesis noise floor for a white series. Stems inside it are indistinguishable from zero. Worth naming because it is the same $1/\sqrt{N}$ that governs every estimator in the curriculum, and because students otherwise read tiny nonzero stems as real structure.

**If the demo misbehaves.** The `pacf` helper runs a fresh least-squares fit per lag, so 25 lags on 3000 samples takes a moment. Reducing `N` makes the fingerprints visibly noisier, which is a good live demonstration of the previous point if you have time.
</details>

## 2. Two Kinds of Memory

💡 **Intuition.** **AR($p$)** — today is a weighted echo of the last $p$ days plus a shock: $x_t = \sum_i \phi_i x_{t-i} + \varepsilon_t$. Memory *recirculates* (IIR!), so the autocorrelation decays gradually, forever. **MA($q$)** — today is a blend of the last $q$ *shocks*: $x_t = \varepsilon_t + \sum_j \theta_j \varepsilon_{t-j}$. Memory is a conveyor belt (FIR!): the ACF cuts to zero *dead* after lag $q$. The **PACF** (correlation at lag $k$ after regressing out lags $1..k{-}1$) mirrors this: AR($p$) cuts off after $p$, MA tails off. Fingerprint table:

| | ACF | PACF |
|---|---|---|
| AR($p$) | tails off | **cuts off at $p$** |
| MA($q$) | **cuts off at $q$** | tails off |

In [2]:
def acf(x, nlags=25):
    x = x - x.mean()
    r = np.correlate(x, x, "full")[len(x)-1:]
    return r[:nlags+1] / r[0]

def pacf(x, nlags=25):
    out = [1.0]
    for k in range(1, nlags+1):
        X = np.column_stack([x[k-j-1:len(x)-j-1] for j in range(k)])
        phi, *_ = np.linalg.lstsq(X, x[k:], rcond=None)
        out.append(phi[-1])
    return np.array(out)

N = 3000
ar2 = sig.lfilter([1], [1, -1.1, 0.5], rng.standard_normal(N))         # AR(2)
ma2 = sig.lfilter([1, 0.8, 0.6], [1], rng.standard_normal(N))          # MA(2)

fig, axes = plt.subplots(2, 2, figsize=(9.5, 4.2))
ci = 1.96/np.sqrt(N)
for row, (name, x_) in zip(axes, [("AR(2)", ar2), ("MA(2)", ma2)]):
    for ax, (fn, ttl) in zip(row, [(acf, "ACF"), (pacf, "PACF")]):
        vals = fn(x_)
        ax.stem(vals, basefmt=" ")
        ax.axhspan(-ci, ci, alpha=0.15, color="gray")
        ax.set_title(f"{name}: {ttl}")
plt.tight_layout(); plt.show()
print("read the fingerprints: AR(2) → PACF cuts at 2;  MA(2) → ACF cuts at 2")

read the fingerprints: AR(2) → PACF cuts at 2;  MA(2) → ACF cuts at 2


/tmp/ipykernel_2047681/1460871271.py:26: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


**What just happened.** Four plots, and the diagonal is what to look at. The AR(2) row shows an ACF that decays gradually across many lags while its PACF has two clear stems and then drops inside the grey band. The MA(2) row is the mirror image: two ACF stems and then nothing, with a PACF that tails off. That is the fingerprint table, generated rather than asserted.

The mechanism is filtering, not statistics. The MA(2) series is white noise through a 3-tap FIR filter, so a sample and its neighbour share shocks only while their windows overlap — beyond lag 2 they have no shock in common, and the correlation is *exactly* zero rather than merely small. The AR(2) series is white noise through an all-pole IIR filter, and feedback means today's shock is still faintly present forever, so the ACF decays geometrically without ever terminating. Everything in the table follows from FIR having finite memory and IIR having infinite memory.

The grey band matters as much as the stems. It marks $\pm 1.96/\sqrt{N}$, the range in which a genuinely white series' sample correlations will wander by chance alone. Stems inside it carry no information — with $N = 3000$ that threshold is about 0.036, so small nonzero values are noise, not structure. Cut $N$ to 200 and the band widens to about 0.14, at which point the MA(2) cut-off becomes a judgement call rather than a reading.

Worth keeping in perspective: these series came from known generators with 3000 clean samples, which is the friendliest case that exists. Real ACF/PACF plots are ambiguous, and identification in practice means fitting several candidate orders and comparing them on held-out data or by an information criterion. The fingerprints are a way to generate good hypotheses, not a decision procedure.

In [3]:
# Fit the AR(2) two ways and recover the coefficients [1.1, −0.5]
# (a) Yule-Walker: autocorrelation + Toeplitz solve
from scipy.linalg import solve_toeplitz
r = acf(ar2, 3)
phi_yw = solve_toeplitz(r[:2], r[1:3])
# (b) conditional least squares: regress x_t on (x_{t-1}, x_{t-2})
X = np.column_stack([ar2[1:-1], ar2[:-2]])
phi_ls, *_ = np.linalg.lstsq(X, ar2[2:], rcond=None)
print(f"Yule-Walker  φ = {phi_yw.round(3)}")
print(f"least squares φ = {phi_ls.round(3)}   (truth [1.1, -0.5])")

Yule-Walker  φ = [ 1.099 -0.497]
least squares φ = [ 1.101 -0.497]   (truth [1.1, -0.5])


**What just happened.** Two estimators, derived from different principles, recover the generating coefficients: Yule–Walker gives $[1.099, -0.497]$ and conditional least squares gives $[1.101, -0.497]$, against a truth of $[1.1, -0.5]$. Both are within about 0.003 — and since we planted the coefficients ourselves, this is an oracle test rather than a plausibility check.

The two routes are worth distinguishing because students often assume they are the same computation. **Yule–Walker** works in the correlation domain: it estimates the ACF, then solves the normal equations $R\phi = r$ — and because $R$ is Toeplitz, `solve_toeplitz` exploits that structure to do it in $O(p^2)$ rather than $O(p^3)$. **Conditional least squares** ignores correlations entirely and regresses $x_t$ directly on its two lags, minimising in-sample prediction error. Different objectives, different code paths, and they agree to the third decimal here because both are consistent estimators and 3000 samples is plenty.

They are not interchangeable in general. Yule–Walker always returns a *stationary* model — the fitted poles land inside the unit circle by construction — which is a real advantage when a forecast is going to be iterated forward, since a non-stationary fit would diverge. Least squares carries no such guarantee and can hand you an explosive model on short or near-unit-root data. Against that, Yule–Walker is more biased in small samples. With $N = 3000$ the choice is immaterial; with $N = 50$ it is not.

Note also what this cell does *not* do: it fits the AR(2) having already been told the order is 2. Order selection is the harder half of the problem, and the PACF plot above is the informal version of it.

---
### 🕐 Session 2 of 2 — *ARIMA: Trends, Fitting & Honest Forecasts* (~40 min)
**Goal:** difference away nonstationarity; forecast with widening uncertainty cones.
**Builds on:** Session 1.

---

<details>
<summary>🎓 <b>Teacher notes — Session 2: ARIMA — Trends, Fitting & Honest Forecasts</b></summary>

**Timing (~40 min).** 8 min why stationarity is required and what breaks without it · 7 min differencing · 15 min the forecast cell, worked slowly · 8 min the cone and what its coverage really says · 2 min buffer.

**Board first.** Plot a random walk freehand and ask for its mean and variance. Neither exists in the fixed sense — the variance grows without bound with $t$ — so "estimate the autocorrelation" is not a well-posed request. Everything in Session 1 quietly assumed stationarity. Differencing is how we buy it back, and framing it as *restoring the hypotheses of the theory we already built* is much better than presenting it as a preprocessing step.

**The landmark to plant.** ARIMA(0,1,0) is the random walk, and its optimal forecast is today's value, flat, forever. That is the persistence baseline from the [RNN bake-off](./Intro_RNN.ipynb). Say plainly that a large fraction of real financial and economic series are very close to this, which is why beating persistence is hard and why the LSTM in that workshop failed to. Students tend to assume a flat forecast means the model has failed; here it can be exactly right.

**Misconception — the big one in this session.** "Differencing removes the trend, so it's just detrending." Not the same operation. Subtracting a fitted straight line assumes a *deterministic* trend that will continue; differencing assumes a *stochastic* trend, a random walk whose direction is not predictable. They imply completely different forecasts — deterministic detrending extrapolates the line confidently, differencing produces a widening cone around a drift. Choosing wrongly produces confident nonsense, and this is the deepest idea in the session.

**Ask the room.** "Why does the interval widen? Why not stay the same width?" Because forecast errors accumulate: predicting two steps ahead means being wrong about step one *and* step two, and those add. Then push further — "how fast does it widen?" For a stationary AR the cone approaches a constant width (the process reverts to its mean, so uncertainty saturates); after differencing, the integration makes it grow like $\sqrt{h}$ without bound. The shape of the cone encodes which world you are in.

**Do not celebrate the 100%.** The printed coverage of a 95% interval is 100%, and it is tempting to present that as success. It is not — it means the intervals are too wide, and the debrief explains why the independence approximation makes them conservative. Being honest here models exactly the scepticism you want students to have about their own results. If you have time, the exact recursion is a genuinely good exercise, and the notebook flags it as one.

**Pacing.** The forecast cell does a lot at once: difference, fit, iterate, accumulate $\psi$-weights, integrate back. Walk it in those five stages rather than reading it top to bottom, and say explicitly that the forecast is computed on the *differenced* scale and then cumulatively summed back — the `train[-1] + np.cumsum(...)` line is where students lose the thread.
</details>

## 3. The I in ARIMA

💡 **Intuition.** AR/MA theory assumes [stationarity](../Intro_DSP/Statistical_Signal_Processing.ipynb) — but real series trend and wander. **Differencing** ($\nabla x_t = x_t - x_{t-1}$) turns a random-walk-with-drift into a stationary series; do it $d$ times and you have ARIMA($p,d,q$): difference, model the stationary residue, un-difference the forecasts. And know thy landmark: ARIMA(0,1,0) is the random walk, whose best forecast is *today's value* — the 'persistence' baseline from the [RNN bake-off](./Intro_RNN.ipynb).

In [4]:
# A trending, wandering series: drift + random walk + AR(2) wiggle
Nw = 700
walk = np.cumsum(0.08 + 0.5*rng.standard_normal(Nw))
wiggle = sig.lfilter([1], [1, -0.6, 0.3], rng.standard_normal(Nw))
y = walk + wiggle

fig, axes = plt.subplots(1, 2, figsize=(9.5, 2.6))
axes[0].plot(y); axes[0].set_title("raw: trending, nonstationary")
axes[1].plot(np.diff(y)); axes[1].set_title("after one difference: stationary, modelable")
plt.tight_layout(); plt.show()

/tmp/ipykernel_2047681/3658402221.py:10: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


**What just happened.** The left panel wanders upward with no fixed level to return to — the mean depends on where you look, so it is not a well-defined quantity and none of Session 1's machinery applies. One difference later, the right panel oscillates around a constant with roughly stable spread. That is the whole of the "I" in ARIMA: a single `np.diff` restored the hypotheses our theory needs.

It works because of how the series was built. `walk` is the cumulative sum of drift plus white noise, and differencing is the exact inverse of cumulative summation — so $\nabla$ returns the increments, which are stationary by construction. The residual wiggle from the AR(2) component survives differencing as something still stationary and still modelable, which is exactly the "stationary residue" that Session 1's tools will now fit.

**Differencing is not detrending, and the difference matters.** Fitting and subtracting a straight line assumes a *deterministic* trend that will keep going; differencing assumes a *stochastic* trend, a random walk with no predictable direction. The two produce very different forecasts from identical-looking data — a confident extrapolated line versus a cone that widens without bound. Our series genuinely is a random walk, so differencing is right here, but on real data this is a modelling decision rather than a mechanical step, and getting it wrong yields forecasts that are confidently wrong rather than honestly uncertain.

One caution to carry into the next cell: differencing is not free. It amplifies high-frequency noise (it is a high-pass filter, with response $|1 - e^{-j\omega}|$ rising with frequency), and differencing an already-stationary series makes the model worse rather than better. The usual discipline is to difference the minimum number of times that achieves stationarity — almost always $d = 1$, occasionally 2, essentially never more.

In [5]:
# ARIMA(2,1,0) by hand: difference → fit AR(2) + mean → forecast → integrate back
train, test = y[:600], y[600:]
dy = np.diff(train)
mu = dy.mean()
X = np.column_stack([dy[1:-1] - mu, dy[:-2] - mu])
phi, *_ = np.linalg.lstsq(X, dy[2:] - mu, rcond=None)
resid = (dy[2:] - mu) - X @ phi
s2 = resid.var()

# iterate the forecast H steps ahead, tracking variance growth
H = len(test)
d_hist = list(dy[-2:] - mu)
fc_d, var_d = [], []
psi = [1.0]                                              # MA(∞) weights of the AR fit
for h_i in range(H):
    fc_d.append(phi[0]*d_hist[-1] + phi[1]*d_hist[-2])
    d_hist.append(fc_d[-1])
    if h_i > 0:
        psi.append(phi[0]*psi[-1] + (phi[1]*psi[-2] if len(psi) > 1 else 0))
    var_d.append(s2 * np.sum(np.square(psi)))
fc_d = np.array(fc_d) + mu
fc = train[-1] + np.cumsum(fc_d)                          # integrate the differences back
var_path = np.cumsum(var_d)                               # variances of a sum of forecast errors (approx: independent)
band = 1.96*np.sqrt(var_path)

plt.figure(figsize=(9, 3))
plt.plot(np.arange(500, 600), train[500:], "k", linewidth=1, label="history")
plt.plot(np.arange(600, 700), test, "k--", linewidth=1, label="future (truth)")
plt.plot(np.arange(600, 700), fc, "C1", label="ARIMA(2,1,0) forecast")
plt.fill_between(np.arange(600, 700), fc-band, fc+band, alpha=0.2, color="C1", label="95% cone")
plt.legend(); plt.title("The honest signature of a good forecast: a cone that widens like √h")
plt.tight_layout(); plt.show()

inside = np.mean(np.abs(test - fc) <= band)
print(f"fraction of future inside the 95% cone: {inside:.0%}")

fraction of future inside the 95% cone: 100%


/tmp/ipykernel_2047681/4091437595.py:32: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


**What just happened.** A full ARIMA(2,1,0) built by hand — difference, fit an AR(2) with a mean term, iterate the forecast 100 steps forward, accumulate the $\psi$-weights to price the uncertainty, and integrate back to the original scale. The truth stays inside the cone, and the forecast tracks the drift rather than the wiggles.

**Read the shape first.** The point forecast decays quickly toward a straight line with slope $\mu$, the mean drift of the differenced series. That is not the model giving up — it is the model being correct. The AR(2) component describes short-lived, mean-reverting wiggle, and after a handful of steps its contribution has decayed to nothing, leaving only the drift. Anything further out is unforecastable by this model, and it says so. A forecaster whose point predictions keep oscillating far into the future is usually hallucinating structure that its own fitted dynamics do not support.

**Now the number, which is a warning rather than a win.** Coverage of the 95% interval came out at **100%**. A well-calibrated 95% interval should miss about 5% of the time — perfect coverage means the cone is too wide, not that the forecast is excellent. The cause is in the code and the notebook flags it: `var_path = np.cumsum(var_d)` treats the $h$-step forecast errors as independent when summing them back to the original scale. They are not independent — they share the same underlying shocks — so the true variance of the integrated forecast is smaller than this sum, and the intervals inherit that conservatism.

This is worth dwelling on because the failure is invisible without the check. The plot looks excellent either way; only computing the coverage reveals the intervals are miscalibrated. The habit to take away is that an uncertainty estimate is a *prediction* about how often you will be wrong, and like any prediction it has to be scored. On one test path with 100 correlated points, incidentally, the empirical coverage is itself a very noisy statistic — the right version of this check runs many simulated paths and looks at the distribution.

**And the honest caveat.** This is the easy case: we generated the data, so the model class is exactly right and only the coefficients had to be learned. Real forecasting has no such guarantee, and misspecification — not parameter error — is usually what dominates. That is precisely why residual checking, mentioned in the next cell, is not optional: whiteness of the residuals is the evidence that the model class was adequate in the first place.

The widening cone is the *point*: a forecaster that doesn't confess growing uncertainty is lying. (The exact interval recursion uses the ψ-weights above; our independence approximation is slightly conservative — a good exercise is deriving the exact one.)

**Model checking:** after fitting, the residuals should be white — run their ACF and a [periodogram](../Intro_DSP/Statistical_Signal_Processing.ipynb); structure left in residuals = model too small.

## 4. Conclusion

ACF/PACF fingerprints identify the model; Yule-Walker/least squares fit it; differencing tames trends; ψ-weights price the uncertainty. This is the baseline that every fancy forecaster must beat — and, as the [RNN workshop](./Intro_RNN.ipynb) showed, often doesn't.

---
## Where next

- [Recurrent Neural Networks](./Intro_RNN.ipynb) — the nonlinear challenger, now with its baseline fully understood.
- [Statistical Signal Processing](../Intro_DSP/Statistical_Signal_Processing.ipynb) — AR spectra: these models as PSD estimators.